In [11]:
import re
import os

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [4]:
df = pd.read_parquet('../prepared_data.parquet').reset_index(drop=True)
for c in ("type", "country", "rating"):
    df[c] = df[c].astype(str)

In [5]:
def to_list(v):
    """List of clean names from an array/list or a comma-separated string."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, str):
        v = v.split(",")
    return [str(x).strip() for x in v if str(x).strip()]

In [6]:
df["genres"] = df["listed_in"].apply(to_list)
df["directors"] = df["director"].apply(to_list)
df["countries"] = df["country"].apply(to_list)
df["decade"] = (df["release_year"] // 10 * 10).astype(str) + "s"

In [7]:
def token(s):
    """'Action & Adventure' -> 'actionadventure'; keeps multi-word names as ONE
    token so 'Martin Scorsese' never matches 'Martin Lawrence'."""
    return re.sub(r"[^a-z0-9]", "", s.lower())
 
 
def make_soup(row, w_genre=3, w_dir=2):
    parts = []
    parts += ["g_" + token(g) for g in row["genres"]] * w_genre   # genres matter most
    parts += ["d_" + token(d) for d in row["directors"]] * w_dir
    parts += ["c_" + token(c) for c in row["countries"]]
    parts += ["r_" + token(row["rating"]), "t_" + token(row["type"]),
              "y_" + row["decade"]]
    return " ".join(parts)
 
 
df["soup"] = df.apply(make_soup, axis=1)

In [8]:
tfidf = TfidfVectorizer(token_pattern=r"\S+", lowercase=False)
X = tfidf.fit_transform(df["soup"])
sim = linear_kernel(X, X)           # rows are L2-normalised -> cosine similarity
np.fill_diagonal(sim, -1)           # never recommend the title to itself
print("Feature matrix:", X.shape, "| similarity matrix:", sim.shape)
 
title_idx = {t.lower(): i for i, t in reversed(list(enumerate(df["title"])))}

Feature matrix: (5915, 4876) | similarity matrix: (5915, 5915)


In [9]:
def recommend(title, k=10):
    i = title_idx.get(title.lower())
    if i is None:
        close = df.loc[df["title"].str.contains(re.escape(title), case=False), "title"]
        raise KeyError(f"'{title}' not found. Did you mean: {list(close.head(5))}")
    top = np.argsort(-sim[i])[:k]
    res = df.loc[top, ["title", "type", "country", "rating", "release_year"]].copy()
    res.insert(0, "genres", df.loc[top, "genres"].apply(", ".join))
    res.insert(0, "similarity", sim[i, top].round(3))
    return res.reset_index(drop=True)

In [14]:
demo = os.environ.get("TITLES", "Dick Johnson Is Dead|Midnight Mass|Ganglands|Sherlock Holmes|Kung Fu Panda")
lines = []
for t in demo.split("|"):
    try:
        r = recommend(t, 5)
    except KeyError as e:
        lines.append(f"### {t}\n{e}\n")
        continue
    i = title_idx[t.lower()]
    lines.append(f"### {df.loc[i, 'title']}  ({', '.join(df.loc[i, 'genres'])})\n")
    lines.append(r.to_markdown(index=False) + "\n")
with open("sample_recommendations.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("\n".join(lines))

### Dick Johnson Is Dead  (Documentaries)

|   similarity | genres        | title                                          | type   | country        | rating   |   release_year |
|-------------:|:--------------|:-----------------------------------------------|:-------|:---------------|:---------|---------------:|
|        0.263 | Documentaries | The Social Dilemma                             | Movie  | United States  | PG-13    |           2020 |
|        0.261 | Documentaries | Jeremy Scott: The People's Designer            | Movie  | United States  | PG-13    |           2015 |
|        0.253 | Documentaries | Lo and Behold: Reveries of the Connected World | Movie  | United States  | PG-13    |           2016 |
|        0.243 | Documentaries | Final Account                                  | Movie  | United Kingdom | PG-13    |           2021 |
|        0.242 | Documentaries | Bully                                          | Movie  | United States  | PG-13    |           2012 |

### 

In [15]:
K = 10
G = [set(g) for g in df["genres"]]
rng = np.random.default_rng(42)
 
 
def jaccard(a, b):
    return len(a & b) / len(a | b)
 
 
def evaluate(sim_matrix, sample, k=K):
    jac, hit, same_type, div = [], [], [], []
    rec_ids = set()
    for i in sample:
        top = np.argsort(-sim_matrix[i])[:k]
        rec_ids.update(top.tolist())
        j = [jaccard(G[i], G[t]) for t in top]
        jac.append(np.mean(j))
        hit.append(np.mean([len(G[i] & G[t]) > 0 for t in top]))
        same_type.append(np.mean(df["type"].values[top] == df["type"].values[i]))
        sub = sim_matrix[np.ix_(top, top)].copy()          # diversity within list
        np.fill_diagonal(sub, np.nan)
        div.append(1 - np.nanmean(np.where(sub < 0, 0, sub)))
    return {
        "genre Jaccard@10": np.mean(jac),
        "genre hit-rate@10": np.mean(hit),
        "same-type rate@10": np.mean(same_type),
        "catalogue coverage": len(rec_ids) / len(df),
    }

In [ ]:
sample = rng.choice(len(df), size=1000, replace=False)
 
# random baseline
rand = rng.random(sim.shape)
np.fill_diagonal(rand, -1)
 
# held-out model: same pipeline without genres in the features
def soup_no_genre(row):
    parts = ["d_" + token(d) for d in row["directors"]] * 2
    parts += ["c_" + token(c) for c in row["countries"]]
    parts += ["r_" + token(row["rating"]), "t_" + token(row["type"]), "y_" + row["decade"]]
    return " ".join(parts)
 
 
X2 = TfidfVectorizer(token_pattern=r"\S+", lowercase=False).fit_transform(
    df.apply(soup_no_genre, axis=1))
sim2 = linear_kernel(X2, X2)
np.fill_diagonal(sim2, -1)
 
rows = {
    "Random baseline": evaluate(rand, sample),
    "Full model (with genres)": evaluate(sim, sample),
    "Held-out (genres removed)": evaluate(sim2, sample),
}
res = pd.DataFrame(rows).T.round(3)
print(res)
with open("evaluation.md", "w", encoding="utf-8") as f:
    f.write(res.to_markdown())
res.to_csv("evaluation.csv")

                           genre Jaccard@10  genre hit-rate@10  \
Random baseline                       0.148              0.391   
Full model (with genres)              0.877              0.982   
Held-out (genres removed)             0.337              0.631   

                           same-type rate@10  catalogue coverage  
Random baseline                        0.932               0.825  
Full model (with genres)               0.997               0.513  
Held-out (genres removed)              0.977               0.401  


AttributeError: 'str' object has no attribute 'write_text'